# ⚡ Alur Kerja Agen Paralel dengan Model GitHub (Python)

## 📋 Tutorial Pemrosesan Paralel Lanjutan

Notebook ini mendemonstrasikan **pola alur kerja paralel** menggunakan Microsoft Agent Framework. Anda akan belajar cara membangun alur kerja pemrosesan paralel berkinerja tinggi di mana beberapa agen AI dieksekusi secara bersamaan, secara dramatis meningkatkan throughput dan memungkinkan proses bisnis multi-threading yang canggih.

## 🎯 Tujuan Pembelajaran

### 🚀 **Dasar-Dasar Pemrosesan Paralel**
- **Eksekusi Agen Paralel**: Jalankan beberapa agen secara bersamaan untuk efisiensi maksimal
- **Orkestrasi Alur Kerja**: Koordinasikan operasi paralel sambil mempertahankan konsistensi data
- **Optimasi Performa**: Capai percepatan signifikan melalui pemrosesan paralel
- **Manajemen Sumber Daya**: Gunakan sumber daya model AI secara efisien di seluruh operasi paralel

### 🏗️ **Pola Paralel Lanjutan**
- **Pemrosesan Fork-Join**: Pisahkan kerja ke beberapa agen dan gabungkan hasilnya
- **Paralelisme Pipeline**: Tahapan eksekusi bertumpuk untuk throughput berkelanjutan
- **Penyeimbangan Beban**: Distribusikan pekerjaan secara merata di seluruh sumber daya agen yang tersedia
- **Titik Sinkronisasi**: Koordinasikan agen paralel pada tahap alur kerja yang kritis

### 🏢 **Aplikasi Paralel Perusahaan**
- **Pemrosesan Dokumen Bervolume Tinggi**: Proses beberapa dokumen secara bersamaan
- **Analisis Konten Waktu Nyata**: Analisis paralel aliran data masuk
- **Optimasi Pemrosesan Batch**: Maksimalkan throughput untuk operasi skala besar
- **Analisis Multi-Mode**: Pemrosesan paralel berbagai jenis konten (teks, gambar, data)

## ⚙️ Persyaratan & Pengaturan

### 📦 **Dependensi yang Diperlukan**

Pasang Agent Framework dengan kemampuan alur kerja paralel:

```bash
pip install agent-framework-core -U
```

### 🔑 **Konfigurasi Model GitHub**

**Pengaturan Lingkungan (.env file):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**Pertimbangan Pemrosesan Paralel:**
- **Batasan Rate**: Pantau batasan rate API Model GitHub untuk permintaan paralel
- **Penggunaan Sumber Daya**: Pertimbangkan penggunaan memori dan CPU dengan beberapa agen paralel
- **Penanganan Kesalahan**: Terapkan pemulihan kesalahan yang tangguh untuk operasi paralel

### 🏗️ **Arsitektur Alur Kerja Paralel**

```mermaid
graph TD
    A[Mulai Workflow] --> B[Eksekusi Bersamaan]
    B --> C[Pool Agen 1]
    B --> D[Pool Agen 2]
    B --> E[Pool Agen 3]
    C --> F[Agregasi Hasil]
    D --> F
    E --> F
    F --> G[Output Akhir]
    
    H[GitHub Models API] --> C
    H --> D
    H --> E
```

**Manfaat Utama:**
- **⚡ Performa**: Percepatan signifikan melalui eksekusi paralel
- **📈 Skalabilitas**: Tangani beban kerja yang meningkat tanpa peningkatan waktu yang proporsional
- **🔄 Efisiensi**: Pemanfaatan sumber daya komputasi yang tersedia menjadi lebih baik
- **🎯 Throughput**: Proses lebih banyak pekerjaan dalam waktu yang sama

## 🎨 **Pola Desain Alur Kerja Paralel**

### 🔍 **Pipeline Riset & Analisis**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Alur Kerja Pemrosesan Data**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Pipeline Pembuatan Konten**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Pemrosesan Multi-Tahap**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Manfaat Performa Perusahaan**

### ⚡ **Optimasi Throughput**
- **Eksekusi Paralel**: Banyak agen bekerja secara bersamaan
- **Pemanfaatan Sumber Daya**: Efisiensi maksimal kapasitas model AI yang tersedia
- **Pengurangan Waktu**: Penurunan signifikan dalam total waktu pemrosesan
- **Arsitektur Skalabel**: Mudah menambah agen paralel sesuai kebutuhan

### 🛡️ **Keandalan & Ketahanan**
- **Toleransi Kesalahan**: Kegagalan agen individu tidak menghentikan seluruh alur kerja
- **Isolasi Kesalahan**: Masalah di satu cabang paralel tidak memengaruhi lainnya
- **Degradasi Terhormat**: Sistem tetap beroperasi meski kapasitas agen berkurang
- **Mekanisme Pemulihan**: Retry otomatis dan penanganan kesalahan untuk operasi yang gagal

### 📊 **Pemantauan & Observabilitas**
- **Pelacakan Eksekusi Paralel**: Pantau kemajuan semua operasi paralel
- **Metode Performa**: Ukur percepatan dan peningkatan efisiensi
- **Analitik Penggunaan Sumber Daya**: Optimalkan alokasi agen paralel
- **Identifikasi Bottleneck**: Temukan dan atasi kendala performa

Mari bangun alur kerja AI paralel berkinerja tinggi! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
